# Projet Fondement IA — Analyse de la Personnalité Client


**Dataset :** `personnalité client_bruité.csv`  
**Objectif :** Analyser, nettoyer et modéliser un jeu de données clients afin de comprendre les profils de personnalité et prédire le comportement d'achat

---

## Sommaire
1. Introduction et contexte métier
2. Importation des librairies
3. Chargement et exploration initiale des données
4. Nettoyage et traitement du bruit (données bruitées)
5. Analyse exploratoire des données (EDA)
6. Feature Engineering
7. Préparation des données pour la modélisation
8. Apprentissage non-supervisé — Segmentation client (Clustering)
9. Apprentissage supervisé — Prédiction de la réponse client
10. Évaluation et comparaison des modèles
11. Interprétation des résultats
12. Conclusion et perspectives

## 1. Introduction et contexte métier

### 1.1 Présentation du problème
Une entreprise souhaite mieux comprendre ses clients pour personnaliser ses campagnes marketing et améliorer son taux de conversion.*

### 1.2 Description du jeu de données
*Présenter brièvement les variables :*
- **Démographie :** `Year_Birth`, `Education`, `Marital_Status`, `Income`, `Kidhome`, `Teenhome`
- **Ancienneté :** `Dt_Customer`, `Recency`
- **Dépenses (2 ans) :** `MntWines`, `MntFruits`, `MntMeatProducts`, `MntFishProducts`, `MntSweetProducts`, `MntGoldProds`
- **Canaux d'achat :** `NumDealsPurchases`, `NumWebPurchases`, `NumCatalogPurchases`, `NumStorePurchases`, `NumWebVisitsMonth`
- **Campagnes :** `AcceptedCmp1`...`AcceptedCmp5`, `Response` (cible), `Complain`

### 1.3 Objectifs
- Nettoyer un dataset bruité
- Réaliser une analyse exploratoire approfondie
- Segmenter les clients en profils types (clustering)
- Prédire la variable cible `Response` (classification supervisée)
- Comparer plusieurs modèles et justifier le meilleur

## 2. Importation des librairies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score

from io import StringIO


## 3. Chargement et exploration initiale des données

### 3.1 Chargement du fichier CSV


In [ ]:

# Chaque ligne du CSV est entourée de guillemets ("..."), ce qui empêche
# pandas de découper correctement sur la tabulation.
with open("personnalité client_bruité.csv", "r", encoding="utf-8") as f:
    contenu = f.read().replace('"', '')

df = pd.read_csv(StringIO(contenu), sep="\t")



### 3.2 Dimensions et types des variables


In [26]:
print("Taille du dataset :") 
print(df.shape)
print("Types des variables :")
print(df.dtypes)
print("Informations sur le dataset :")
print(df.info())


Taille du dataset :
(2240, 29)
Types des variables :
ID                       int64
Year_Birth               int64
Education               object
Marital_Status          object
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer             object
Recency                float64
MntWines               float64
MntFruits                int64
MntMeatProducts        float64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds           float64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Z_CostContact            int64
Z_Revenue                int64
Response                 int64
dtype: object
Informations sur le dataset :
<cla

In [25]:
print("Aperçu des données :" )
print(df.head())

Aperçu des données :
     ID  Year_Birth   Education Marital_Status   Income  Kidhome  Teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293.0        1         0   

  Dt_Customer  Recency  MntWines  ...  NumWebVisitsMonth  AcceptedCmp3  \
0  04-09-2012     58.0     635.0  ...                  7             0   
1  08-03-2014      NaN      11.0  ...                  5             0   
2  21-08-2013     26.0     426.0  ...                  4             0   
3  10-02-2014     26.0      11.0  ...                  6             0   
4  19-01-2014     94.0     173.0  ...                  5             0   

   AcceptedCmp4  AcceptedCmp5  AcceptedCmp1  AcceptedCmp2  Compla

### 3.3 Statistiques descriptives

In [28]:
print("Statistiques descriptives :")
print(df.describe())


Statistiques descriptives :
                 ID   Year_Birth         Income      Kidhome     Teenhome  \
count   2240.000000  2240.000000    2216.000000  2240.000000  2240.000000   
mean    5592.159821  1968.805804   52247.251354     0.444196     0.506250   
std     3246.662198    11.984069   25173.076661     0.538398     0.544538   
min        0.000000  1893.000000    1730.000000     0.000000     0.000000   
25%     2828.250000  1959.000000   35303.000000     0.000000     0.000000   
50%     5458.500000  1970.000000   51381.500000     0.000000     0.000000   
75%     8427.750000  1977.000000   68522.000000     1.000000     1.000000   
max    11191.000000  1996.000000  666666.000000     2.000000     2.000000   

           Recency     MntWines    MntFruits  MntMeatProducts  \
count  2239.000000  2239.000000  2240.000000      2239.000000   
mean     49.114337   303.899955    26.302232       167.019652   
std      28.967970   336.668329    39.773434       225.741716   
min       0.000000

### 3.4 Premier diagnostic du bruit


In [ ]:

#print("Valeurs manquantes :")
#print(df.isnull().sum())
print("pourcentage de valeurs manquantes :")
print(df.isnull().mean() * 100)

#la colonne Income etant a 1% de valeurs manquantes, on la garde. 

pourcentage de valeurs manquantes :
ID                     0.000000
Year_Birth             0.000000
Education              0.000000
Marital_Status         0.000000
Income                 1.071429
Kidhome                0.000000
Teenhome               0.000000
Dt_Customer            0.000000
Recency                0.044643
MntWines               0.044643
MntFruits              0.000000
MntMeatProducts        0.044643
MntFishProducts        0.000000
MntSweetProducts       0.000000
MntGoldProds           0.044643
NumDealsPurchases      0.000000
NumWebPurchases        0.000000
NumCatalogPurchases    0.000000
NumStorePurchases      0.000000
NumWebVisitsMonth      0.000000
AcceptedCmp3           0.000000
AcceptedCmp4           0.000000
AcceptedCmp5           0.000000
AcceptedCmp1           0.000000
AcceptedCmp2           0.000000
Complain               0.000000
Z_CostContact          0.000000
Z_Revenue              0.000000
Response               0.000000
dtype: float64


In [32]:

print("Doublons :")
print(df.duplicated().sum())


Doublons :
0


In [33]:

print("Modalités douteuses :")
for col in df.columns:
    print(f"{col} : {df[col].unique()}")


Modalités douteuses :
ID : [5524 2174 4141 ... 7270 8235 9405]
Year_Birth : [1957 1954 1965 1984 1981 1967 1971 1985 1974 1950 1983 1976 1959 1952
 1987 1946 1980 1949 1982 1979 1951 1969 1986 1989 1963 1970 1973 1943
 1975 1996 1968 1964 1977 1978 1955 1966 1988 1948 1958 1972 1960 1945
 1991 1962 1953 1961 1956 1992 1900 1893 1990 1947 1899 1993 1994 1941
 1944 1995 1940]
Education : ['Graduation' 'PhD' 'Master' 'Basic' '2n Cycle']
Marital_Status : ['Single' 'Together' 'Married' 'Divorced' 'Widow' 'Alone' 'Absurd' 'YOLO']
Income : [58138. 46344. 71613. ... 56981. 69245. 52869.]
Kidhome : [0 1 2]
Teenhome : [0 1 2]
Dt_Customer : ['04-09-2012' '08-03-2014' '21-08-2013' '10-02-2014' '19-01-2014'
 '09-09-2013' '13-11-2012' '08-05-2013' '06-06-2013' '13-03-2014'
 '15-11-2013' '10-10-2012' '24-11-2012' '24-12-2012' '31-08-2012'
 '28-03-2013' '03-11-2012' '08-08-2012' '06-01-2013' '23-12-2012'
 '11-01-2014' '18-03-2013' '02-01-2013' '27-05-2013' '20-02-2013'
 '31-05-2013' '22-11-2013' '22-0

## 4. Nettoyage et traitement du bruit

### 4.1 Suppression des doublons

In [ ]:
# À compléter : suppression des doublons


### 4.2 Gestion des valeurs manquantes
*Stratégie à justifier (suppression, imputation par moyenne / médiane / mode, KNNImputer...). En particulier, traiter `Income` qui contient beaucoup de NaN.*

In [ ]:
# À compléter : traitement des valeurs manquantes (justifier la stratégie choisie)


### 4.3 Correction des modalités catégorielles
*Harmoniser les modalités (ex. "YOLO", "Absurd" dans `Marital_Status`, casse, espaces...).*

In [ ]:
# À compléter : harmoniser / corriger les modalités catégorielles


### 4.4 Détection et traitement des outliers
*Boxplots, méthode IQR ou z-score sur `Income`, `Year_Birth` (âge irréaliste), `MntWines`, etc.*

In [ ]:
# À compléter : détection et traitement des outliers


### 4.5 Conversion des types
*Convertir `Dt_Customer` en datetime, vérifier les types numériques, gérer les colonnes constantes (`Z_CostContact`, `Z_Revenue`).*

In [ ]:
# À compléter : conversion des types et suppression des colonnes inutiles


## 5. Analyse exploratoire des données (EDA)

### 5.1 Distribution des variables numériques
*Histogrammes, KDE plots, boxplots pour `Income`, `MntWines`, `Recency`...*

In [ ]:
# À compléter : visualisation des distributions numériques


### 5.2 Distribution des variables catégorielles
*Diagrammes en barres pour `Education`, `Marital_Status`, `Response`.*

In [ ]:
# À compléter : visualisation des variables catégorielles


### 5.3 Analyse bivariée et matrice de corrélation
*Heatmap des corrélations, pairplots, relations entre `Income` et les dépenses.*

In [ ]:
# À compléter : matrice de corrélation et analyses bivariées


### 5.4 Analyse de la variable cible `Response`
*Étudier le déséquilibre de classe et les relations cible / features.*

In [ ]:
# À compléter : étude de la variable cible et de son équilibre


## 6. Feature Engineering

### 6.1 Création de nouvelles variables
Variables suggérées :
- `Age` = année courante − `Year_Birth`
- `Anciennete_jours` = aujourd'hui − `Dt_Customer`
- `Total_Spending` = somme des `Mnt*`
- `Total_Children` = `Kidhome` + `Teenhome`
- `Total_AcceptedCmp` = somme des campagnes acceptées
- `Family_Size` = couple/seul + enfants
- `Has_Children` (binaire)

In [ ]:
# À compléter : création des nouvelles variables (Age, Anciennete, Total_Spending, ...)


### 6.2 Encodage des variables catégorielles
*One-Hot Encoding ou Label Encoding pour `Education` et `Marital_Status`.*

In [ ]:
# À compléter : encodage des variables catégorielles


## 7. Préparation des données pour la modélisation

### 7.1 Sélection des features et de la cible

In [ ]:
# À compléter : définir X (features) et y (cible Response)


### 7.2 Séparation train / test

In [ ]:
# À compléter : split train/test (penser à stratify sur la cible)


### 7.3 Standardisation / mise à l'échelle
*Justifier la nécessité d'une standardisation pour les modèles sensibles aux échelles (KNN, régression logistique, K-Means).*

In [ ]:
# À compléter : appliquer un scaler (fit sur train, transform sur train et test)


## 8. Apprentissage non supervisé — Segmentation client (Clustering)

### 8.1 Réduction de dimension (PCA) pour la visualisation

In [ ]:
# À compléter : ACP / PCA pour réduire la dimension


### 8.2 Choix du nombre de clusters
*Méthode du coude (Elbow) et score de silhouette.*

In [ ]:
# À compléter : déterminer le nombre optimal de clusters (Elbow + Silhouette)


### 8.3 Application du K-Means et visualisation des clusters

In [ ]:
# À compléter : entraîner le K-Means et visualiser les clusters dans le plan PCA


### 8.4 Profilage des segments
*Décrire chaque cluster (revenu moyen, dépenses, âge, structure familiale...) afin d'identifier des personae clients.*

In [ ]:
# À compléter : statistiques par cluster et description des profils


## 9. Apprentissage supervisé — Prédiction de la réponse client

### 9.1 Modèle de référence (baseline)
*Régression logistique ou DummyClassifier pour servir de point de comparaison.*

In [ ]:
# À compléter : modèle baseline


### 9.2 Arbre de décision

In [ ]:
# À compléter : Decision Tree


### 9.3 K-Nearest Neighbors (KNN)

In [ ]:
# À compléter : KNN


### 9.4 Random Forest

In [ ]:
# À compléter : Random Forest


### 9.5 Gradient Boosting

In [ ]:
# À compléter : Gradient Boosting


### 9.6 Optimisation des hyperparamètres
*GridSearchCV ou RandomizedSearchCV sur le(s) meilleur(s) modèle(s).*

In [ ]:
# À compléter : recherche d'hyperparamètres


## 10. Évaluation et comparaison des modèles

### 10.1 Métriques de classification
*Accuracy, Précision, Rappel, F1-score, AUC. Discuter du choix de la métrique pertinente compte tenu du déséquilibre de la cible.*

In [ ]:
# À compléter : tableau récapitulatif des métriques pour tous les modèles


### 10.2 Matrices de confusion et courbes ROC

In [ ]:
# À compléter : afficher les matrices de confusion et tracer les courbes ROC


### 10.3 Validation croisée
*Vérifier la stabilité du meilleur modèle.*

In [ ]:
# À compléter : cross_val_score sur le modèle retenu


## 11. Interprétation des résultats

### 11.1 Importance des variables (feature importance)
*Identifier les variables les plus discriminantes pour prédire `Response`.*

In [ ]:
# À compléter : afficher l'importance des features


### 11.2 Croisement clustering / classification
*Les segments identifiés sont-ils cohérents avec la propension à répondre à une campagne ?*

In [ ]:
# À compléter : analyse croisée segments x Response


### 11.3 Recommandations métier
*Tirer des conclusions exploitables : sur quelle cible concentrer la prochaine campagne, quel canal privilégier...*

## 12. Conclusion et perspectives

### 12.1 Synthèse
*Rappeler la problématique, le pipeline mis en place et les principaux résultats obtenus.*

### 12.2 Limites du travail
*Bruit résiduel, déséquilibre de classe, taille du jeu de données, choix arbitraires de seuils...*

### 12.3 Perspectives d'amélioration
- Tester d'autres algorithmes (XGBoost, SVM, réseaux de neurones)
- Rééquilibrage de classe (SMOTE, class_weight)
- Analyse RFM ou Customer Lifetime Value
- Mise en production / dashboard interactif